# 1. TCGA-BRCA preprocessing

Same steps as `scripts/01_preprocess.py`, one cell at a time, so you can look at what the
composition adjustment actually does before trusting it.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import Config
from src.io_utils import data_path, read_matrix, read_table, results_path

cfg = Config.load("../config.json")

In [ ]:
from src.preprocessing import (
    clean_symbols, composition_scores, drop_constant, filter_low_expression,
    residualize, select_variable_genes,
)

expr = read_matrix(data_path("tcga", "HiSeqV2.gz"))
expr = expr.loc[:, [c for c in expr.columns if c.endswith("-01")]]
clinical = read_table(data_path("tcga", "BRCA_clinicalMatrix.tsv"), index_col=0)
survival = read_table(data_path("tcga", "BRCA_survival.tsv"), index_col=0)
expr.shape

Values are already log2(norm_count+1), so no second log transform.

In [ ]:
subtype = clinical["PAM50Call_RNAseq"]
keep = [s for s in expr.columns if s in subtype.index and pd.notna(subtype.get(s))]
expr = expr.loc[:, keep]
subtype = subtype.loc[keep]
subtype.value_counts()

In [ ]:
expr = drop_constant(filter_low_expression(clean_symbols(expr), cfg.min_count, cfg.min_samples))
expr.shape

## Tissue composition

Breast tumour blocks carry a lot of fat and stroma. Unadjusted, those genes are among the
most variable in the cohort and take over the network.

In [ ]:
composition = composition_scores(expr)
composition.corr().round(2)

In [ ]:
unadjusted = select_variable_genes(expr, cfg.n_network_genes)
adipose = [g for g in ["ADIPOQ", "FABP4", "PLIN1", "CIDEC", "AQP7", "CIDEA"] if g in unadjusted]
print(len(unadjusted), "genes selected;", len(adipose), "of them adipocyte markers:", adipose)

In [ ]:
adjusted = residualize(expr, composition[cfg.composition_axes])
genes = select_variable_genes(adjusted, cfg.n_network_genes)
print(len(set(genes) & set(unadjusted)), "of", cfg.n_network_genes, "genes survive the adjustment")
print("still adipocyte:", [g for g in adipose if g in genes])

## Survival table

One row per patient, TCGA column names, so METABRIC can be coerced into the same shape
later.

In [ ]:
from src.survival import prepare_endpoint

meta = pd.DataFrame({"subtype": subtype})
for col in ["OS", "OS.time", "DSS", "DSS.time", "DFI", "DFI.time", "PFI", "PFI.time"]:
    meta[col] = survival[col].reindex(meta.index)

surv = prepare_endpoint(meta, cfg.survival_endpoint, cfg.max_followup_days)
print(len(surv), "patients,", int(surv["event"].sum()), "events")
surv.head()

Run `python scripts/01_preprocess.py` to write all of this to `results/preprocessed/`.